In [ ]:
import numpy as np
#sys.path.append('/home/joao/lib/dredge/dredge-python/')
from pathlib import Path
from labdata.schema import *
from labdata import chronic_paper as cp
import matplotlib.pyplot as plt

In [ ]:
# target dates for chronic timepoint comparison
from datetime import timedelta
COMPARISON_TIMEPOINTS = [timedelta(days=7), timedelta(days=28), timedelta(days=49), timedelta(days=70)]

In [ ]:
chronic_insertions = cp.IBLMatchedInsertion()
chronic_recording_keys = []
for ins in chronic_insertions:
    recordings = (EphysRecording() & ins) * Session()

    for timepoint in COMPARISON_TIMEPOINTS:
        target_datetime = ins['procedure_datetime'] + timepoint
        recordings_with_time_diff = recordings.proj(days_from_target='ABS(TIMESTAMPDIFF(DAY, session_datetime, "{}"))'.format(target_datetime.strftime('%Y-%m-%d %H:%M:%S')))
        key = recordings_with_time_diff.fetch(order_by='days_from_target ASC', as_dict=True, limit=1)[0]
        if key['days_from_target'] > 3:
            print(f"{key['days_from_target']} away from target date {timepoint}")
        chronic_recording_keys.append(key)
chronic_probe_recordings = EphysRecording.proj() & chronic_recording_keys
acute_probe_recordings = EphysRecording.proj() & cp.IBLMatchedInsertion().to_ephys_session()

In [ ]:
# launch spike extraction for all recordings
#all_probe_sessions = EphysRecording.ProbeSetting() & (acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY'))
all_probe_sessions =  (EphysRecording() & acute_probe_recordings.fetch('KEY') + chronic_probe_recordings.fetch('KEY')) - cp.DredgeSpikeDetection()
labdata_submission_commands = []
t = []
for k in all_probe_sessions.fetch(order_by='subject_name, session_name'):
    dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]} --force-submit'
    t.append(dict(session_name=k['session_name']))
    #dd = f'labdata2 run detect -t aws -a {k["subject_name"]} -s {k["session_name"]}'
    labdata_submission_commands.append(dd)
    #os.system(dd) # uncomment to run
labdata_submission_commands = np.unique(labdata_submission_commands)
print(f'There are {len(labdata_submission_commands)} sessions to process.')
print('\n'.join(labdata_submission_commands))

all_probe_session_keys = all_probe_sessions.fetch('KEY')

In [ ]:
# get the probe keys so we can run dredge on each probe-session
acute_session_probe_keys = EphysRecording.ProbeSetting() & (cp.IBLMatchedInsertion().EphysRecording().proj('session_name', 
                                                                                                           'dataset_name',
                                                                                                           chronic_mouse='subject_name',
                                                                                                           chronic_probe_id='probe_id',
                                                                                                           subject_name='matched_subject_name',
                                                                                                           probe_num='matched_probe_num'))
chronic_session_probe_keys = (EphysRecording.ProbeSetting() & chronic_probe_recordings)
chronic_probes = dj.U('subject_name', 'probe_num') & chronic_session_probe_keys # sessions have the same trajectories 

In [ ]:
# now run dredge algorithm on chronic subjects
subs, probenums = chronic_probes.fetch('subject_name', 'probe_num')
i = 0
first_session_per_mouse_probe = chronic_probes.aggr(chronic_session_probe_keys, first_session='MIN(session_name)').fetch(as_dict=True)
sess = first_session_per_mouse_probe[i]
cp.DredgeSpikeDetection().plot_raster(sess['subject_name'], sess['first_session'], sess['probe_num'], shank_num=0)


In [ ]:
# now run dredge on acute subjects

In [ ]:
mindepth = 250
maxdepth = 3500
mindepth = 0
maxdepth = 3600
#maxdepth = 2900
#cp.DredgeMotionEstimate().populate_subject(k['subject_name'], k['probe_num'], dredge_params_id=0, n_workers=4, min_spike_depth=[mindepth], max_spike_depth=[maxdepth])
#(cp.DredgeMotionEstimate() & k).delete()

In [ ]:
DREDGE_PARAMS_ID = 1
# TODO: these probably need the probe num included
acute_dredge = cp.DredgeMotionEstimate() & acute_probe_recordings & {'dredge_params_id': DREDGE_PARAMS_ID}
chronic_dredge = cp.DredgeMotionEstimate() & chronic_probe_recordings & {'dredge_params_id': DREDGE_PARAMS_ID}
chronic_dredge, acute_dredge


In [ ]:
#from matplotlib import gridspec
#spec = gridspec.GridSpec(ncols=1, nrows=2,
#                         wspace=.2,
#                         hspace=.1,height_ratios=[2,.5])
#
##data = acute_dredge.fetch(as_dict=True)
#data = chronic_dredge.fetch(as_dict=True)
#d = data[-6]
#
## apply low pass filter to data
#from scipy.signal import butter, filtfilt
#def lowpass_filter(data, cutoff_freq, fs, order=4):
#    nyquist = 0.5 * fs
#    normal_cutoff = cutoff_freq / nyquist
#    b, a = butter(order, normal_cutoff, btype='low', analog=False)
#    return filtfilt(b, a, data)
#
#fs = np.mean(np.diff(d['time_bin_centers_s']))
#filtered = lowpass_filter(d['displacement'], cutoff_freq=.01, fs=fs, order=4)
#
#fig = plt.figure(figsize=(12,8))
#ax = fig.add_subplot(spec[0])
#plt.gca().spines[['right', 'top']].set_visible(False)
#
#cp.DredgeSpikeDetection().plot_raster(shank_num=0, subject_name=d['subject_name'], session_name=d['session_name'], probe_num=d['probe_num'],
#                                      clim=(0,70), cmap='gray_r',rasterized=True)
#plt.plot(d['time_bin_centers_s'], d['displacement'].T + d['spatial_bin_centers_um'])
#plt.ylabel('Depth along shank (um)')
##plt.gca().spines[['right', 'top']].set_visible(False)
#
#fig.add_subplot(spec[1], sharex=ax)
#plt.plot(d['time_bin_centers_s'], d['displacement'].T)
#plt.plot(d['time_bin_centers_s'], filtered.T)
#plt.ylabel('Drift estimate (um)')
#plt.ylim(-10,10)
#plt.gca().spines[['right', 'top']].set_visible(False)

In [ ]:
# apply low pass filter to data
from scipy.signal import butter, filtfilt
def lowpass_filter(data, cutoff_freq, fs, order=4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
cmap = plt.get_cmap('Set3')
import seaborn as sns
cmap = sns.color_palette("hls", len(acute_dredge))

N_SAMPS = 30

all_data = []
unique_chronic_insertions = pd.DataFrame(chronic_dredge.fetch('subject_name','probe_num', as_dict=True)).value_counts().index.values
for i,(sub,prb) in enumerate(unique_chronic_insertions):
    # do chronic insertions
    tss, displacements = (Session * chronic_dredge & dict(subject_name=sub, probe_num=prb)).fetch('time_bin_centers_s','displacement', order_by='session_datetime')
    chronic_drifts = []
    for ts, displacement in zip(tss, displacements):
        fs = np.mean(np.diff(ts))
        filtered = lowpass_filter(displacement, cutoff_freq=.01, fs=fs, order=4)
        #cumulative_drifts = np.sum(np.abs(np.diff(filtered, axis=1)), axis=1)
        cumulative_drifts = np.abs(np.mean(filtered[:,:N_SAMPS], axis=1) - np.mean(filtered[:,-N_SAMPS:], axis=1))
        avg_cumulative_drift = np.mean(cumulative_drifts)
        chronic_drifts.append(avg_cumulative_drift)

    # now do matched acute insertion
    acute_session = (cp.IBLMatchedInsertion & (ProbeInsertion * EphysRecording.ProbeSetting & chronic_dredge & dict(subject_name=sub, probe_num=prb))).to_ephys_session()
    ts, displacement = (acute_dredge & acute_session).fetch1('time_bin_centers_s','displacement')
    fs = np.mean(np.diff(ts))
    filtered = lowpass_filter(displacement, cutoff_freq=.01, fs=fs, order=4)
    #cumulative_drifts = np.sum(np.abs(np.diff(filtered, axis=1)), axis=1)
    cumulative_drifts = np.abs(np.mean(filtered[:,:N_SAMPS], axis=1) - np.mean(filtered[:,-N_SAMPS:], axis=1))
    acute_cumulative_drift = np.mean(cumulative_drifts)
    combined_data = np.insert(chronic_drifts, 0, acute_cumulative_drift)
    all_data.append(combined_data)
    #plt.plot(np.arange(len(combined_data)), combined_data, label=f"{sub} {prb}", color=cmap[i], marker='o', markersize=8)
    plt.plot(np.arange(len(combined_data)), combined_data, label=f"{sub} {prb}", color=cmap[i], marker='o', markersize=4, alpha=.6)
all_data = np.stack(all_data)
mean = np.mean(all_data, axis=0)
sems = np.std(all_data, axis=0) / np.sqrt(all_data.shape[0])
#plt.errorbar(np.arange(len(combined_data)) +.1, mean, yerr=sems, fmt='o', color='black', label='mean ± sem', markersize=4)
plt.errorbar(np.arange(len(combined_data)) +.1, mean, yerr=sems, fmt='o', color='black', label='mean ± sem', markersize=6)


plt.xlim(-.5, len(combined_data) - .5)
plt.xticks(np.arange(5), ['paired acute insertion'] + [tp.days for tp in COMPARISON_TIMEPOINTS], rotation=0)
plt.xlabel('Days post insertion')
plt.ylabel('Cumulative drift (um)')
plt.legend()